# Connectors Example through LLama Stack

This notebook demonstrates how to work with connectors within LLama stack and its Responses API Implementation.
This example has the following setup going:

- LLama stack server running on `localhost:8321`
- Kubernetes MCP server running on `localhost:8080`
- Slack MCP server running on `localhost:13080`
- **OpenAI API key** (see "API key" section below; never commit the key to the repo)

## MCP server set up:

Follow the instructions from the [Kubernetes MCP server](https://github.com/opendatahub-io/agents/blob/main/examples/kubernetes-mcp/README.md) to set up the Kubernetes MCP server and kubernetes cluster.and

Follow the instructions from the [Slack MCP server](https://github.com/opendatahub-io/agents/blob/main/examples/slack-mcp/README.md) to set up the Slack MCP server.

## LLama stack server set up

Add the connectors store and connectors configuration to the `run.yaml` that your llama stack server instance will use as follows:

```yaml
  stores:
    ...
    connectors:
      namespace: connectors
      backend: kv_default
```

```yaml
connectors:
- connector_id: kubernetes
  url: "http://localhost:8080/mcp"
  connector_type: mcp
- connector_id: slack
  url: "http://localhost:13080/sse"
  connector_type: mcp
  ...
```


### Complete `run.yaml` for rerference:

```yaml
version: 2
image_name: llamastack-crimson
container_image: null
apis:
- inference
- agents
- safety
- vector_io
- files
- tool_runtime
providers:
  inference:
  - provider_id: openai
    provider_type: remote::openai
    config:
      api_key: ${env.OPENAI_API_KEY}
    module: null
  agents:
  - provider_id: meta-reference
    provider_type: inline::meta-reference
    config:
      persistence:
        agent_state:
          namespace: agents
          backend: kv_default
        responses:
          table_name: responses
          backend: sql_default
          max_write_queue_size: 10000
          num_writers: 4
  files:
  - provider_id: meta-reference-files
    provider_type: inline::localfs
    config:
      storage_dir: ${env.FILES_STORAGE_DIR:=~/.llama/crimson/files}
      metadata_store:
        table_name: files_metadata
        backend: sql_default
storage:
  backends:
    kv_default:
      type: kv_sqlite
      db_path: ${env.SQLITE_STORE_DIR:=~/.llama/distributions/starter}/kvstore.db
    sql_default:
      type: sql_sqlite
      db_path: ${env.SQLITE_STORE_DIR:=~/.llama/distributions/starter}/sql_store.db
  stores:
    metadata:
      namespace: registry
      backend: kv_default
    inference:
      table_name: inference_store
      backend: sql_default
      max_write_queue_size: 10000
      num_writers: 4
    conversations:
      table_name: openai_conversations
      backend: sql_default
    prompts:
      namespace: prompts
      backend: kv_default
    connectors:
      namespace: connectors
      backend: kv_default
connectors:
- connector_id: kubernetes
  url: "http://localhost:8080/mcp"
  connector_type: mcp
- connector_id: slack
  url: "http://localhost:13080/sse"
  connector_type: mcp
models:
- model_id: gpt-4o
  model_type: llm
  provider_id: openai
shields: []
vector_dbs: []
datasets: []
scoring_fns: []
benchmarks: []
tool_groups: []
logging: null
server:
  port: 8321
  auth: null
  tls_certfile: null
  tls_keyfile: null
  tls_cafile: null
  host: null
  quota: null
  cors: null

```

## Querying the API

LLama stack exposes a read-only API that allows users to discover confgured connectors, and gather information about specific MCP servers and the tools offered by that server 

### List available connectors:

In [11]:
!curl -s http://localhost:8321/v1alpha/connectors | python -m json.tool

{
    "data": [
        {
            "connector_type": "mcp",
            "connector_id": "kubernetes",
            "url": "http://localhost:8080/mcp",
            "server_label": null,
            "server_name": null,
            "server_description": null,
            "server_version": null
        },
        {
            "connector_type": "mcp",
            "connector_id": "slack",
            "url": "http://localhost:13080/sse",
            "server_label": null,
            "server_name": null,
            "server_description": null,
            "server_version": null
        }
    ]
}


### GET specfic conenctor:

In [12]:
!curl -s http://localhost:8321/v1alpha/connectors/kubernetes | python -m json.tool

{
    "connector_type": "mcp",
    "connector_id": "kubernetes",
    "url": "http://localhost:8080/mcp",
    "server_label": null,
    "server_name": "kubernetes-mcp-server",
    "server_description": null,
    "server_version": "v0.0.57"
}


### List all tools from a given connector:

In [13]:
!curl -s http://localhost:8321/v1alpha/connectors/slack/tools | python -m json.tool

{
    "data": [
        {
            "toolgroup_id": null,
            "name": "channels_list",
            "description": "Get list of channels",
            "input_schema": {
                "type": "object",
                "properties": {
                    "channel_types": {
                        "description": "Comma-separated channel types. Allowed values: 'mpim', 'im', 'public_channel', 'private_channel'. Example: 'public_channel,private_channel,im'",
                        "type": "string"
                    },
                    "cursor": {
                        "description": "Cursor for pagination. Use the value of the last row and column in the response as next_cursor field returned from the previous request.",
                        "type": "string"
                    },
                    "limit": {
                        "default": 100,
                        "description": "The maximum number of items to return. Must be an integer between 1 and 1000 (maxi

### Get a specific tool's information:

In [16]:
!curl -s http://localhost:8321/v1alpha/connectors/kubernetes/tools/namespaces_list | python -m json.tool

{
    "toolgroup_id": null,
    "name": "namespaces_list",
    "description": "List all the Kubernetes namespaces in the current cluster",
    "input_schema": {
        "type": "object",
        "properties": {
            "context": {
                "type": "string",
                "description": "Optional parameter selecting which context to run the tool in. Defaults to kind-kind if not set",
                "enum": [
                    "/api-agentic-mcp-jolf-p3-openshiftapps-com:443/jrao",
                    "default/api-agentic-mcp-i6a3-p3-openshiftapps-com:443/cluster-admin",
                    "jrao/api-agentic-mcp-i6a3-p3-openshiftapps-com:443/cluster-admin",
                    "kind-kind",
                    "test-jd/api-agentic-mcp-jolf-p3-openshiftapps-com:443/jrao"
                ]
            }
        }
    },
    "output_schema": null,
    "metadata": {
        "endpoint": "http://localhost:8080/mcp"
    }
}


## Referencing connectors in Responses API Requests:

Users may use this feature to pass references to MCP servers by passing in a `connector_id` instead of the MCP server's full URL. LLama stack internally resolves the `connector_id` to invoke the correct MCP server and perform the tool call.

In [23]:
import json

def _format_args(args):
    if args is None:
        return "None"
    if isinstance(args, dict):
        return json.dumps(args, indent=2)
    if isinstance(args, str):
        try:
            return json.dumps(json.loads(args), indent=2)
        except (json.JSONDecodeError, TypeError):
            return args
    return str(args)

def pretty_print_result(title, resp):
    print(f"\n{'='*60}\n{title}\n{'='*60}")
    output_text = getattr(resp, "output_text", None)
    if output_text:
        print("Output text:")
        print(output_text)
    if hasattr(resp, "output"):
        for i, item in enumerate(resp.output or []):
            if getattr(item, "type", None) == "mcp_call":
                args = getattr(item, "arguments", None)
                print(f"\n--- mcp_call[{i}] ---")
                print(f"  server:   {getattr(item, 'server_label', '')}")
                print(f"  tool:     {getattr(item, 'name', '')}")
                print("  arguments:")
                for line in _format_args(args).splitlines():
                    print(f"    {line}")

In [26]:
import os
from openai import OpenAI

# Optional: load OPENAI_API_KEY from a local .env file (gitignored)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Client for Llama Stack Responses API (key from env; never hardcode)
client = OpenAI(
    base_url="http://localhost:8321/v1",
    api_key=os.getenv("OPENAI_API_KEY") or "not-needed",
)


In [27]:
resp = client.responses.create(
        model="openai/gpt-4o",
        tools=[
            {
                "type": "mcp",
                "server_label": "slack",
                "connector_id": "slack",
                "require_approval": "never",
            },
            {
                "type": "mcp",
                "server_label": "kubernetes",
                "connector_id": "kubernetes",
                "require_approval": "never",
            }

        ],
        input=[{
            "role": "user",
            "content": (
                "return all the pods in the 'kube-system' namespace and send it to the slack channel #platform-notifications"
            )
        }],
    )
pretty_print_result("Result", resp)


Result
Output text:
The list of pods currently running in the 'kube-system' namespace has been successfully sent to the Slack channel **#platform-notifications**.

--- mcp_call[2] ---
  server:   kubernetes
  tool:     pods_list_in_namespace
  arguments:
    {
      "namespace": "kube-system"
    }

--- mcp_call[3] ---
  server:   slack
  tool:     channels_list
  arguments:
    {
      "channel_types": "public_channel"
    }

--- mcp_call[4] ---
  server:   slack
  tool:     conversations_add_message
  arguments:
    {
      "channel_id": "C09M96QGVB9",
      "payload": "Here are the pods currently running in the 'kube-system' namespace:\n\n| NAMESPACE   | NAME                                       | READY | STATUS  | RESTARTS | AGE  | IP         | NODE               | LABELS |\n|-------------|--------------------------------------------|-------|---------|----------|------|------------|--------------------|--------|\n| kube-system | coredns-66bc5c9577-dtt7n                   | 1/1   

![Slack message](Screenshot%202026-02-06%20at%207.15.18 PM.png)